## 1. Setup and Configuration

In [ ]:
# Import required libraries
from databricks import sql
import pandas as pd
import numpy as np
import os
from dotenv import load_dotenv
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
# Load environment variables from parent directory
load_dotenv(dotenv_path='../.env')

# Validate credentials
required_vars = ['DATABRICKS_SERVER_HOSTNAME', 'DATABRICKS_HTTP_PATH', 'DATABRICKS_TOKEN']
missing = [var for var in required_vars if not os.getenv(var)]
if missing:
    raise ValueError(f"Missing environment variables: {', '.join(missing)}")

print("✓ Environment configured")

## 2. Analysis Parameters

In [ ]:
# Customer and time period settings
customer_filter = 'cds_8007'  # SAPPORO DRUG JP
start_date = '2023-10-01'
end_date = '2024-03-31'

# Product filters
category_filter = 'Laundry'
target_condition = "jp_sub_brand_alter_lang_name IN ('アリエール', 'ボールド')"  # Target products

# Next purchase analysis settings
next_product_granularity = 'jp_brand_alter_lang_name'  # Level to track next purchases
max_days_between = 90  # Maximum days to consider for "next" purchase

print(f"✓ Parameters set")
print(f"  Analysis period: {start_date} to {end_date}")
print(f"  Category: {category_filter}")
print(f"  Target: {target_condition}")
print(f"  Next purchase tracked at: {next_product_granularity}")
print(f"  Max days between purchases: {max_days_between}")

## 3. Build and Execute Query

In [ ]:
# Build next purchase analysis query
query = f"""
WITH base_transactions AS (
    SELECT
        idpos.shopper_key AS shopper_id,
        CAST(idpos.sales_period_group_end_date_part AS DATE) AS purchase_date,
        jp_brand_alter_lang_name AS brand,
        jp_sub_brand_alter_lang_name AS sub_brand,
        {next_product_granularity} AS product_name,
        pos_unit_sales_qty AS unit,
        pos_sales_amt AS value,
        CASE WHEN {target_condition} THEN 1 ELSE 0 END AS is_target_product
    FROM
        cdl_customer_prod.gold_customer_loyalty.loyalty_transact_fct_v1_vw idpos
        LEFT JOIN id_pos_ai_1.prod_dim_ext_vw prod ON idpos.prod_key = prod.prod_key
        LEFT JOIN id_pos_ai_1.shopper_dim_generic_vw shopper ON idpos.shopper_key = shopper.shopper_key
    WHERE
        jp_category_name = '{category_filter}'
        AND idpos.data_provider_code_part = '{customer_filter}'
        AND sales_period_group_end_date_part BETWEEN '{start_date}' AND '{end_date}'
        AND shopper.member_ind = 'Y'
        AND {next_product_granularity} IS NOT NULL
),
target_purchases AS (
    SELECT
        shopper_id,
        purchase_date AS target_purchase_date,
        product_name AS target_product,
        value AS target_value,
        unit AS target_unit
    FROM base_transactions
    WHERE is_target_product = 1
),
all_purchases_with_sequence AS (
    SELECT
        base.shopper_id,
        base.purchase_date,
        base.product_name,
        base.value,
        base.unit,
        target.target_purchase_date,
        target.target_product,
        DATEDIFF(base.purchase_date, target.target_purchase_date) AS days_after_target,
        ROW_NUMBER() OVER (
            PARTITION BY target.shopper_id, target.target_purchase_date 
            ORDER BY base.purchase_date
        ) AS purchase_sequence
    FROM base_transactions base
    INNER JOIN target_purchases target 
        ON base.shopper_id = target.shopper_id
        AND base.purchase_date > target.target_purchase_date
        AND base.purchase_date <= DATE_ADD(target.target_purchase_date, {max_days_between})
),
next_purchases AS (
    SELECT
        shopper_id,
        target_purchase_date,
        target_product,
        purchase_date AS next_purchase_date,
        product_name AS next_product,
        value AS next_value,
        unit AS next_unit,
        days_after_target
    FROM all_purchases_with_sequence
    WHERE purchase_sequence = 1
)
SELECT
    target_product,
    next_product,
    COUNT(*) AS transition_count,
    COUNT(DISTINCT shopper_id) AS unique_shoppers,
    AVG(days_after_target) AS avg_days_to_next,
    MIN(days_after_target) AS min_days_to_next,
    MAX(days_after_target) AS max_days_to_next,
    SUM(next_value) AS total_next_value,
    SUM(next_unit) AS total_next_units,
    AVG(next_value) AS avg_next_value
FROM next_purchases
GROUP BY target_product, next_product
ORDER BY transition_count DESC
"""

print("✓ SQL query constructed")
print(f"  Query length: {len(query)} characters")

In [ ]:
# Execute query
with sql.connect(
    server_hostname=os.getenv("DATABRICKS_SERVER_HOSTNAME"),
    http_path=os.getenv("DATABRICKS_HTTP_PATH"),
    access_token=os.getenv("DATABRICKS_TOKEN")
) as connection:
    with connection.cursor() as cursor:
        cursor.execute(query)
        result = cursor.fetchall()
        columns = [desc[0] for desc in cursor.description]
        df = pd.DataFrame(result, columns=columns)

# Convert data types
df['transition_count'] = df['transition_count'].astype('Int64')
df['unique_shoppers'] = df['unique_shoppers'].astype('Int64')
df['avg_days_to_next'] = pd.to_numeric(df['avg_days_to_next'], errors='coerce')
df['min_days_to_next'] = pd.to_numeric(df['min_days_to_next'], errors='coerce')
df['max_days_to_next'] = pd.to_numeric(df['max_days_to_next'], errors='coerce')
df['total_next_value'] = pd.to_numeric(df['total_next_value'], errors='coerce')
df['total_next_units'] = pd.to_numeric(df['total_next_units'], errors='coerce')
df['avg_next_value'] = pd.to_numeric(df['avg_next_value'], errors='coerce')

print(f"✓ Query executed successfully")
print(f"  Retrieved {len(df)} transition patterns")
print(f"  Total transitions tracked: {df['transition_count'].sum():,}")
print(f"\nFirst few rows:")
df.head(10)

## 4. Process and Analyze Results

In [ ]:
# Calculate transition probabilities
total_by_target = df.groupby('target_product')['transition_count'].sum().to_dict()
df['transition_probability'] = df.apply(
    lambda row: (row['transition_count'] / total_by_target[row['target_product']] * 100) 
    if row['target_product'] in total_by_target else 0, 
    axis=1
)

# Identify repurchase vs switch patterns
df['is_repurchase'] = df['target_product'] == df['next_product']
repurchase_rate = df[df['is_repurchase']]['transition_count'].sum() / df['transition_count'].sum() * 100

# Summary statistics
total_transitions = df['transition_count'].sum()
unique_target_products = df['target_product'].nunique()
unique_next_products = df['next_product'].nunique()
avg_time_to_next = df['avg_days_to_next'].mean()

print("=" * 60)
print("NEXT PURCHASE ANALYSIS SUMMARY")
print("=" * 60)
print(f"\nOverall Metrics:")
print(f"  Total Transitions: {total_transitions:,}")
print(f"  Unique Target Products: {unique_target_products}")
print(f"  Unique Next Products: {unique_next_products}")
print(f"  Average Days to Next Purchase: {avg_time_to_next:.1f}")
print(f"  Repurchase Rate: {repurchase_rate:.1f}%")

# Top next purchase patterns
print(f"\nTop 10 Next Purchase Patterns:")
top_10 = df.nlargest(10, 'transition_count')
for idx, row in top_10.iterrows():
    repurchase_flag = "(REPURCHASE)" if row['is_repurchase'] else ""
    print(f"  {row['target_product']:25s} → {row['next_product']:25s} {repurchase_flag}")
    print(f"    Count: {row['transition_count']:5,} | Probability: {row['transition_probability']:5.1f}% | Avg Days: {row['avg_days_to_next']:.1f}")

## 5. Visualizations

In [ ]:
# Top next purchase transitions (bar chart)
top_15 = df.nlargest(15, 'transition_count').copy()
top_15['transition_label'] = top_15['target_product'] + ' → ' + top_15['next_product']
top_15['color'] = top_15['is_repurchase'].map({True: 'Repurchase', False: 'Switch'})

fig = px.bar(
    top_15,
    x='transition_count',
    y='transition_label',
    orientation='h',
    title='Top 15 Next Purchase Patterns',
    labels={'transition_count': 'Number of Transitions', 'transition_label': 'Purchase Pattern'},
    text='transition_count',
    color='color',
    color_discrete_map={'Repurchase': '#32CD32', 'Switch': '#FF8C00'}
)

fig.update_traces(texttemplate='%{text:,}', textposition='outside')
fig.update_layout(
    yaxis={'categoryorder':'total ascending'},
    height=600,
    showlegend=True
)

fig.show()

In [ ]:
# Repurchase vs Switch breakdown
behavior_summary = df.groupby('is_repurchase').agg({
    'transition_count': 'sum',
    'unique_shoppers': 'sum'
}).reset_index()
behavior_summary['behavior'] = behavior_summary['is_repurchase'].map({True: 'Repurchase', False: 'Switch'})

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('By Transitions', 'By Unique Shoppers'),
    specs=[[{'type':'pie'}, {'type':'pie'}]]
)

colors = ['#32CD32', '#FF8C00']

fig.add_trace(
    go.Pie(
        labels=behavior_summary['behavior'],
        values=behavior_summary['transition_count'],
        marker_colors=colors,
        texttemplate='%{label}<br>%{value:,}<br>(%{percent})',
        hole=0.3
    ),
    row=1, col=1
)

fig.add_trace(
    go.Pie(
        labels=behavior_summary['behavior'],
        values=behavior_summary['unique_shoppers'],
        marker_colors=colors,
        texttemplate='%{label}<br>%{value:,}<br>(%{percent})',
        hole=0.3
    ),
    row=1, col=2
)

fig.update_layout(
    title_text="Repurchase vs Brand Switching Behavior",
    height=400
)

fig.show()

In [ ]:
# Time to next purchase distribution
time_distribution = df.copy()
time_distribution['days_bucket'] = pd.cut(
    time_distribution['avg_days_to_next'],
    bins=[0, 7, 14, 30, 60, 90],
    labels=['0-7 days', '8-14 days', '15-30 days', '31-60 days', '61-90 days']
)

bucket_summary = time_distribution.groupby('days_bucket')['transition_count'].sum().reset_index()

fig = px.bar(
    bucket_summary,
    x='days_bucket',
    y='transition_count',
    title='Time to Next Purchase Distribution',
    labels={'days_bucket': 'Days After Target Purchase', 'transition_count': 'Number of Transitions'},
    text='transition_count',
    color='transition_count',
    color_continuous_scale='Blues'
)

fig.update_traces(texttemplate='%{text:,}', textposition='outside')
fig.update_layout(height=500, showlegend=False)

fig.show()

In [ ]:
# Transition probability heatmap (for top products)
# Get top 10 target and next products
top_targets = df.groupby('target_product')['transition_count'].sum().nlargest(10).index.tolist()
top_nexts = df.groupby('next_product')['transition_count'].sum().nlargest(10).index.tolist()

heatmap_data = df[df['target_product'].isin(top_targets) & df['next_product'].isin(top_nexts)].copy()
pivot_table = heatmap_data.pivot_table(
    index='target_product',
    columns='next_product',
    values='transition_probability',
    fill_value=0
)

fig = px.imshow(
    pivot_table,
    title='Transition Probability Heatmap (Top 10 Products)',
    labels=dict(x="Next Product", y="Target Product", color="Probability %"),
    color_continuous_scale='YlOrRd',
    aspect='auto'
)

fig.update_layout(height=600)
fig.show()

## 6. Data Tables

In [ ]:
# Top switching patterns (excluding repurchase)
print("Top 20 Brand Switching Patterns:")
switching_df = df[~df['is_repurchase']].nlargest(20, 'transition_count')[[
    'target_product', 'next_product', 'transition_count', 'transition_probability', 
    'avg_days_to_next', 'unique_shoppers'
]]
switching_df

In [ ]:
# Repurchase patterns
print("\nRepurchase Patterns:")
repurchase_df = df[df['is_repurchase']].sort_values('transition_count', ascending=False)[[
    'target_product', 'transition_count', 'transition_probability', 
    'avg_days_to_next', 'unique_shoppers'
]]
repurchase_df

## 7. Data Export

In [ ]:
# Export to Excel (optional)
export_file = f"next_purchase_analysis_{category_filter}_{start_date}_to_{end_date}.xlsx"

with pd.ExcelWriter(export_file, engine='openpyxl') as writer:
    # Summary sheet
    summary_df = pd.DataFrame({
        'Metric': [
            'Total Transitions',
            'Unique Target Products',
            'Unique Next Products',
            'Average Days to Next Purchase',
            'Repurchase Rate %'
        ],
        'Value': [
            f"{total_transitions:,}",
            f"{unique_target_products}",
            f"{unique_next_products}",
            f"{avg_time_to_next:.1f}",
            f"{repurchase_rate:.1f}%"
        ]
    })
    summary_df.to_excel(writer, sheet_name='Summary', index=False)
    
    # All transitions
    df.to_excel(writer, sheet_name='All_Transitions', index=False)
    
    # Switching patterns
    switching_df.to_excel(writer, sheet_name='Switching_Patterns', index=False)
    
    # Repurchase patterns
    repurchase_df.to_excel(writer, sheet_name='Repurchase_Patterns', index=False)

print(f"✓ Data exported to: {export_file}")